In [5]:
import numpy as np
import pandas as pd
from collections import defaultdict

# Load dataset
df = pd.read_csv("train_hate.csv")
sentences = [sentence.split() for sentence in df["Sentence"]]

# Build vocabulary
word_freq = defaultdict(int)
for sentence in sentences:
    for word in sentence:
        word_freq[word] += 1

vocab = list(word_freq.keys())
word_to_index = {word: i for i, word in enumerate(vocab)}
vocab_size = len(vocab)
embedding_size = 100

# Initialize embeddings and context weights
embeddings = np.random.randn(vocab_size, embedding_size) * 0.01  # Small initial values
context_weights = np.random.randn(vocab_size, embedding_size) * 0.01

# Hyperparameters
window_size = 2
learning_rate = 0.01
epochs = 5
negative_samples = 5  # Key fix: Add negative sampling

# Training loop
# for epoch in range(epochs):
#     total_loss = 0
#     for sentence in sentences:
#         for i, target_word in enumerate(sentence):
#             target_idx = word_to_index[target_word]
            
#             # Get context words
#             start = max(0, i - window_size)
#             end = min(len(sentence), i + window_size + 1)
#             context_words = sentence[start:i] + sentence[i+1:end]
            
#             # Update for each context word
#             for context_word in context_words:
#                 context_idx = word_to_index[context_word]
                
#                 # Positive sample update
#                 score = np.dot(embeddings[target_idx], context_weights[context_idx])
#                 sigmoid = 1 / (1 + np.exp(-score))
#                 grad = (sigmoid - 1) * learning_rate  # Gradient for positive sample
                
#                 embeddings[target_idx] -= grad * context_weights[context_idx]
#                 context_weights[context_idx] -= grad * embeddings[target_idx]
                
#                 # Negative sampling
#                 for _ in range(negative_samples):
#                     neg_idx = np.random.randint(0, vocab_size)
#                     while neg_idx == context_idx:  # Ensure negative sample != context
#                         neg_idx = np.random.randint(0, vocab_size)
                    
#                     neg_score = np.dot(embeddings[target_idx], context_weights[neg_idx])
#                     neg_sigmoid = 1 / (1 + np.exp(-neg_score))
#                     grad_neg = (neg_sigmoid - 0) * learning_rate  # Gradient for negative sample
                    
#                     embeddings[target_idx] -= grad_neg * context_weights[neg_idx]
#                     context_weights[neg_idx] -= grad_neg * embeddings[target_idx]
    
#     print(f"Epoch {epoch+1}/{epochs} completed")

# Save embeddings
np.save("embeddings.npy", embeddings)

Epoch 1/5 completed
Epoch 2/5 completed
Epoch 3/5 completed
Epoch 4/5 completed
Epoch 5/5 completed


In [6]:
def get_sentence_embedding(sentence, embeddings, word_to_index):
    word_indices = [word_to_index[word] for word in sentence if word in word_to_index]
    if not word_indices:
        return np.zeros(embedding_size)
    return np.mean(embeddings[word_indices], axis=0)

# Convert sentences to embeddings
X = np.array([get_sentence_embedding(sentence, embeddings, word_to_index) for sentence in sentences])
y = df["Tag"].values  # Ensure `y` is defined

In [7]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

# Define the model
model = Sequential([
    Dense(64, activation='relu', input_shape=(embedding_size,)),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])

# Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Calculate class weights
class_counts = df["Tag"].value_counts()
class_weights = {0: 1, 1: class_counts[0] / class_counts[1]}  # Penalize misclassifying Hate (1)

# Train the model
model.fit(X, y, epochs=15, batch_size=32, validation_split=0.2, class_weight=class_weights)
model.save("hate_classifier.keras")

C:\Users\shrey\AppData\Roaming\Python\Python312\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/15
92/92 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.4909 - loss: 0.8953 - val_accuracy: 0.6189 - val_loss: 0.6877
Epoch 2/15
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.4729 - loss: 0.8753 - val_accuracy: 0.6148 - val_loss: 0.6759
Epoch 3/15
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.5039 - loss: 0.8769 - val_accuracy: 0.5423 - val_loss: 0.6927
Epoch 4/15
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.5026 - loss: 0.8663 - val_accuracy: 0.5888 - val_loss: 0.6905
Epoch 5/15
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.5435 - loss: 0.8630 - val_accuracy: 0.6325 - val_loss: 0.6871
Epoch 6/15
92/92 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.4842 - loss: 0.8701 - val_accuracy: 0.6298 - val_loss: 0.6798
Epoch 7/15
92/92 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5756 - loss: 0.8572 - val_accuracy: 0.3852 - val_loss: 0.7078
Epoch 8/15
92/92 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5217 - loss: 0.8692 - val_accuracy: 0.4126 - val_loss:

In [8]:
# Load validation data
val_df = pd.read_csv("val_hate.csv")
val_sentences = [sentence.split() for sentence in val_df["Sentence"]]

# Generate validation embeddings
X_val = np.array([get_sentence_embedding(sentence, embeddings, word_to_index) for sentence in val_sentences])
y_val = val_df["Tag"].values

# Predict probabilities
y_pred_probs = model.predict(X_val)

# Convert probabilities to binary labels (0 or 1)
y_pred = (y_pred_probs > 0.3).astype(int)  # Adjusted threshold

# Calculate F1 score and classification report
from sklearn.metrics import f1_score, classification_report  # Ensure this line is included

f1 = f1_score(y_val, y_pred)
print(f"F1 Score: {f1:.4f}")

# Detailed report (precision, recall, F1 per class)
print(classification_report(y_val, y_pred))

15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step
F1 Score: 0.4893
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       309
           1       0.32      1.00      0.49       148

    accuracy                           0.32       457
   macro avg       0.16      0.50      0.24       457
weighted avg       0.10      0.32      0.16       457



C:\Users\shrey\AppData\Roaming\Python\Python312\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\shrey\AppData\Roaming\Python\Python312\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\shrey\AppData\Roaming\Python\Python312\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} i

In [10]:
import numpy as np
import pandas as pd
from collections import defaultdict

# Load dataset
df = pd.read_csv("sarcasm_train.csv")
sentences = [sentence.split() for sentence in df["Sentence"]]

# Build vocabulary
word_freq = defaultdict(int)
for sentence in sentences:
    for word in sentence:
        word_freq[word] += 1

vocab = list(word_freq.keys())
word_to_index = {word: i for i, word in enumerate(vocab)}
vocab_size = len(vocab)
embedding_size = 100

# Initialize embeddings and context weights
embeddings = np.random.randn(vocab_size, embedding_size) * 0.01  # Small initial values
context_weights = np.random.randn(vocab_size, embedding_size) * 0.01

# Hyperparameters
window_size = 2
learning_rate = 0.01
epochs = 5
negative_samples = 5  # Key fix: Add negative sampling

# Training loop
# for epoch in range(epochs):
#     total_loss = 0
#     for sentence in sentences:
#         for i, target_word in enumerate(sentence):
#             target_idx = word_to_index[target_word]
            
#             # Get context words
#             start = max(0, i - window_size)
#             end = min(len(sentence), i + window_size + 1)
#             context_words = sentence[start:i] + sentence[i+1:end]
            
#             # Update for each context word
#             for context_word in context_words:
#                 context_idx = word_to_index[context_word]
                
#                 # Positive sample update
#                 score = np.dot(embeddings[target_idx], context_weights[context_idx])
#                 sigmoid = 1 / (1 + np.exp(-score))
#                 grad = (sigmoid - 1) * learning_rate  # Gradient for positive sample
                
#                 embeddings[target_idx] -= grad * context_weights[context_idx]
#                 context_weights[context_idx] -= grad * embeddings[target_idx]
                
#                 # Negative sampling
#                 for _ in range(negative_samples):
#                     neg_idx = np.random.randint(0, vocab_size)
#                     while neg_idx == context_idx:  # Ensure negative sample != context
#                         neg_idx = np.random.randint(0, vocab_size)
                    
#                     neg_score = np.dot(embeddings[target_idx], context_weights[neg_idx])
#                     neg_sigmoid = 1 / (1 + np.exp(-neg_score))
#                     grad_neg = (neg_sigmoid - 0) * learning_rate  # Gradient for negative sample
                    
#                     embeddings[target_idx] -= grad_neg * context_weights[neg_idx]
#                     context_weights[neg_idx] -= grad_neg * embeddings[target_idx]
    
#     print(f"Epoch {epoch+1}/{epochs} completed")

# Save embeddings
np.save("embeddings_sarcasm.npy", embeddings)

def get_sentence_embedding(sentence, embeddings, word_to_index):
    word_indices = [word_to_index[word] for word in sentence if word in word_to_index]
    if not word_indices:
        return np.zeros(embedding_size)
    return np.mean(embeddings[word_indices], axis=0)

# Convert sentences to embeddings
X = np.array([get_sentence_embedding(sentence, embeddings, word_to_index) for sentence in sentences])
y = df["Tag"].values  # Ensure `y` is defined

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

# Define the model
model = Sequential([
    Dense(64, activation='relu', input_shape=(embedding_size,)),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])

# Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Calculate class weights
class_counts = df["Tag"].value_counts()
class_weights = {0: 1, 1: class_counts[0] / class_counts[1]}  # Penalize misclassifying Hate (1)

# Train the model
model.fit(X, y, epochs=15, batch_size=32, validation_split=0.2, class_weight=class_weights)
model.save("sarcasm_classifier.keras")


# Load validation data
val_df = pd.read_csv("sarcasm_val.csv")
val_sentences = [sentence.split() for sentence in val_df["Sentence"]]

# Generate validation embeddings
X_val = np.array([get_sentence_embedding(sentence, embeddings, word_to_index) for sentence in val_sentences])
y_val = val_df["Tag"].values

# Predict probabilities
y_pred_probs = model.predict(X_val)

# Convert probabilities to binary labels (0 or 1)
y_pred = (y_pred_probs > 0.3).astype(int)  # Adjusted threshold

# Calculate F1 score and classification report
from sklearn.metrics import f1_score, classification_report  # Ensure this line is included

f1 = f1_score(y_val, y_pred)
print(f"F1 Score: {f1:.4f}")

# Detailed report (precision, recall, F1 per class)
print(classification_report(y_val, y_pred))

Epoch 1/5 completed
Epoch 2/5 completed
Epoch 3/5 completed
Epoch 4/5 completed
Epoch 5/5 completed
Epoch 1/15


C:\Users\shrey\AppData\Roaming\Python\Python312\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


105/105 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.5892 - loss: 1.2944 - val_accuracy: 0.8488 - val_loss: 0.6561
Epoch 2/15
105/105 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5969 - loss: 1.2914 - val_accuracy: 0.8583 - val_loss: 0.6358
Epoch 3/15
105/105 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6458 - loss: 1.3211 - val_accuracy: 0.8893 - val_loss: 0.5693
Epoch 4/15
105/105 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7883 - loss: 1.2217 - val_accuracy: 0.7512 - val_loss: 0.6758
Epoch 5/15
105/105 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6439 - loss: 1.2235 - val_accuracy: 0.8381 - val_loss: 0.6257
Epoch 6/15
105/105 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5314 - loss: 1.2783 - val_accuracy: 0.7952 - val_loss: 0.6560
Epoch 7/15
105/105 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6874 - loss: 1.2068 - val_accuracy: 0.7512 - val_loss: 0.6700
Epoch 8/15
105/105 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5335 - loss: 1.2369 - val_accuracy: 0.7976 - val_

In [11]:
import numpy as np
import pandas as pd
from collections import defaultdict

# Load dataset
df = pd.read_csv("humor_train.csv")
sentences = [sentence.split() for sentence in df["Sentence"]]

# Build vocabulary
word_freq = defaultdict(int)
for sentence in sentences:
    for word in sentence:
        word_freq[word] += 1

vocab = list(word_freq.keys())
word_to_index = {word: i for i, word in enumerate(vocab)}
vocab_size = len(vocab)
embedding_size = 100

# Initialize embeddings and context weights
embeddings = np.random.randn(vocab_size, embedding_size) * 0.01  # Small initial values
context_weights = np.random.randn(vocab_size, embedding_size) * 0.01

# Hyperparameters
window_size = 2
learning_rate = 0.01
epochs = 5
negative_samples = 5  # Key fix: Add negative sampling

# Training loop
# for epoch in range(epochs):
#     total_loss = 0
#     for sentence in sentences:
#         for i, target_word in enumerate(sentence):
#             target_idx = word_to_index[target_word]
            
#             # Get context words
#             start = max(0, i - window_size)
#             end = min(len(sentence), i + window_size + 1)
#             context_words = sentence[start:i] + sentence[i+1:end]
            
#             # Update for each context word
#             for context_word in context_words:
#                 context_idx = word_to_index[context_word]
                
#                 # Positive sample update
#                 score = np.dot(embeddings[target_idx], context_weights[context_idx])
#                 sigmoid = 1 / (1 + np.exp(-score))
#                 grad = (sigmoid - 1) * learning_rate  # Gradient for positive sample
                
#                 embeddings[target_idx] -= grad * context_weights[context_idx]
#                 context_weights[context_idx] -= grad * embeddings[target_idx]
                
#                 # Negative sampling
#                 for _ in range(negative_samples):
#                     neg_idx = np.random.randint(0, vocab_size)
#                     while neg_idx == context_idx:  # Ensure negative sample != context
#                         neg_idx = np.random.randint(0, vocab_size)
                    
#                     neg_score = np.dot(embeddings[target_idx], context_weights[neg_idx])
#                     neg_sigmoid = 1 / (1 + np.exp(-neg_score))
#                     grad_neg = (neg_sigmoid - 0) * learning_rate  # Gradient for negative sample
                    
#                     embeddings[target_idx] -= grad_neg * context_weights[neg_idx]
#                     context_weights[neg_idx] -= grad_neg * embeddings[target_idx]
    
#     print(f"Epoch {epoch+1}/{epochs} completed")

# Save embeddings
np.save("embeddings_humor.npy", embeddings)

def get_sentence_embedding(sentence, embeddings, word_to_index):
    word_indices = [word_to_index[word] for word in sentence if word in word_to_index]
    if not word_indices:
        return np.zeros(embedding_size)
    return np.mean(embeddings[word_indices], axis=0)

# Convert sentences to embeddings
X = np.array([get_sentence_embedding(sentence, embeddings, word_to_index) for sentence in sentences])
y = df["Tag"].values  # Ensure `y` is defined

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

# Define the model
model = Sequential([
    Dense(64, activation='relu', input_shape=(embedding_size,)),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])

# Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Calculate class weights
class_counts = df["Tag"].value_counts()
class_weights = {0: 1, 1: class_counts[0] / class_counts[1]}  # Penalize misclassifying Hate (1)

# Train the model
model.fit(X, y, epochs=15, batch_size=32, validation_split=0.2, class_weight=class_weights)
model.save("humor_classifier.keras")


# Load validation data
val_df = pd.read_csv("humor_val.csv")
val_sentences = [sentence.split() for sentence in val_df["Sentence"]]

# Generate validation embeddings
X_val = np.array([get_sentence_embedding(sentence, embeddings, word_to_index) for sentence in val_sentences])
y_val = val_df["Tag"].values

# Predict probabilities
y_pred_probs = model.predict(X_val)

# Convert probabilities to binary labels (0 or 1)
y_pred = (y_pred_probs > 0.3).astype(int)  # Adjusted threshold

# Calculate F1 score and classification report
from sklearn.metrics import f1_score, classification_report  # Ensure this line is included

f1 = f1_score(y_val, y_pred)
print(f"F1 Score: {f1:.4f}")

# Detailed report (precision, recall, F1 per class)
print(classification_report(y_val, y_pred))

Epoch 1/5 completed
Epoch 2/5 completed
Epoch 3/5 completed
Epoch 4/5 completed
Epoch 5/5 completed
Epoch 1/15


C:\Users\shrey\AppData\Roaming\Python\Python312\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


59/59 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.5206 - loss: 0.5602 - val_accuracy: 0.3729 - val_loss: 0.7075
Epoch 2/15
59/59 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.4548 - loss: 0.5663 - val_accuracy: 0.6504 - val_loss: 0.6821
Epoch 3/15
59/59 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5350 - loss: 0.5592 - val_accuracy: 0.3729 - val_loss: 0.7004
Epoch 4/15
59/59 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.4460 - loss: 0.5684 - val_accuracy: 0.6377 - val_loss: 0.6813
Epoch 5/15
59/59 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5777 - loss: 0.5600 - val_accuracy: 0.3729 - val_loss: 0.7015
Epoch 6/15
59/59 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.4464 - loss: 0.5620 - val_accuracy: 0.3729 - val_loss: 0.6980
Epoch 7/15
59/59 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5019 - loss: 0.5606 - val_accuracy: 0.3729 - val_loss: 0.6948
Epoch 8/15
59/59 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.4604 - loss: 0.5617 - val_accuracy: 0.6398 - val_loss: 0.6883
Epo

C:\Users\shrey\AppData\Roaming\Python\Python312\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\shrey\AppData\Roaming\Python\Python312\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\shrey\AppData\Roaming\Python\Python312\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} i